# CoLM — LLMs with Notebooks

Run this notebook in Google Colab to start CoLM. You'll get:
- A Jupyter-like notebook web UI
- An AI assistant sidebar with tool use (code execution, cell management, file browsing)
- A public URL via ngrok

**Requirements:**
- API keys for any providers you want to use (see Secrets cell below) — free tiers available on OpenRouter, Groq, GLM, and more
- An [ngrok auth token](https://dashboard.ngrok.com/get-started/your-authtoken) (free tier available)

In [1]:
# Install Node.js 20+ if not already present
import subprocess, sys, os

try:
    result = subprocess.run(['node', '--version'], capture_output=True, text=True, check=True)
    version = result.stdout.strip()
    print(f'Node.js already installed: {version}')
    major = int(version.lstrip('v').split('.')[0])
    if major < 18:
        print('Version too old, upgrading...')
        raise Exception('old version')
except:
    print('Installing Node.js 20...')
    !curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1
    !apt-get install -y nodejs > /dev/null 2>&1
    !node --version

Node.js already installed: v20.19.0


In [2]:
# Clone the CoLM repository
import os
if not os.path.isdir('colm'):
    !git clone https://github.com/reecdev/colm.git
%cd colm

/content/colm


In [3]:
# Install npm dependencies and build the frontend
!npm install
!npm run build

⠙⠹⠸⠼⠴⠦
up to date, audited 104 packages in 2s
⠦
⠦36 packages are looking for funding
⠧  run `npm fund` for details
⠧
found 0 vulnerabilities
⠧⠙
> colm@1.0.0 build
> esbuild src/frontend/main.jsx --bundle --outfile=public/bundle.js --loader:.jsx=jsx

⠙
  public/bundle.js  2.2mb ⚠️

⚡ Done in 628ms
⠙

In [4]:
# Install ngrok
!pip install -q pyngrok
!which ngrok || (curl -s https://ngrok-agent.s3.amazonaws.com/ngrok.asc | tee /etc/apt/trusted.gpg.d/ngrok.asc >/dev/null && echo 'deb https://ngrok-agent.s3.amazonaws.com buster main' | tee /etc/apt/sources.list.d/ngrok.list >/dev/null && apt-get update -qq && apt-get install -y -qq ngrok)

/usr/local/bin/ngrok


In [5]:
# Set your API keys — you'll be prompted for any that aren't already set
from google.colab import userdata
import getpass

provider_keys = {
    'OpenRouter': 'OPENROUTER_API_KEY',
    'OpenAI': 'OPENAI_API_KEY',
    'Anthropic': 'ANTHROPIC_API_KEY',
    'Google': 'GOOGLE_API_KEY',
    'GLM': 'GLM_API_KEY',
    'Groq': 'GROQ_API_KEY',
    'OpenCode Zen': 'OPENCODE_API_KEY',
    'HuggingFace': 'HUGGINGFACE_API_KEY',
}

ngrok_token = os.environ.get('NGROK_TOKEN')
if not ngrok_token:
    try:
        ngrok_token = userdata.get('NGROK_TOKEN')
    except Exception:
        ngrok_token = None
if not ngrok_token:
    ngrok_token = getpass.getpass('ngrok auth token (press Enter to skip): ')
if ngrok_token:
    os.environ['NGROK_TOKEN'] = ngrok_token

set_any = False
for label, env_name in provider_keys.items():
    val = os.environ.get(env_name)
    if not val:
        try:
            val = userdata.get(env_name)
        except Exception:
            val = None
    if not val:
        val = getpass.getpass(f'{label} API key (press Enter to skip): ')
    if val:
        os.environ[env_name] = val
        set_any = True

print('Keys set successfully' if set_any else 'Warning: No API keys set')

Keys set successfully


In [ ]:
# Start CoLM server and ngrok tunnel
import subprocess
import threading
import time
import os

# Start ngrok first
tunnel = None
public_url = None

if ngrok_token:
    from pyngrok import ngrok, conf
    conf.get_default().auth_token = ngrok_token
    tunnel = ngrok.connect(15050)
    public_url = tunnel.public_url
    print(f'ngrok tunnel established: {public_url}')

# Start the Node.js server
server_proc = subprocess.Popen(
    ['node', 'bin/cli.js'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env={
        **os.environ,
        'PORT': '15050',
        **({'BASE_URL': public_url, 'PUBLIC_URL': public_url} if public_url else {})
    }
)

time.sleep(2)

if public_url:
    print(f'\nCoLM is running at: {public_url}')
else:
    print('\nNo ngrok token — CoLM running at: http://localhost:15050')

for line in iter(server_proc.stdout.readline, ''):
    print(line, end='')

print('\nServer logs (Ctrl+C to stop):')
print('=' * 50)

ngrok tunnel established: https://bacteria-gully-groggily.ngrok-free.dev

CoLM is running at: http://bacteria-gully-groggily.ngrok-free.dev
Web UI running at: http://localhost:15050
Press Ctrl+C to stop.
